## PRACTICAL IMPLEMENTATION OF SEVERAL WELL KNOWN STREAM CIPHERS

##  ChaCha20 – Using `cryptography` 

The following script demonstrates how to encrypt and decrypt data using the ChaCha20 stream cipher via Python's cryptography library, and then manually reconstructs and verifies the keystream using the XOR property.

 First, we import the necessary cryptographic primitives and generate our secret key, nonce, and plaintext message.
  

In [1]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms
from cryptography.hazmat.backends import default_backend
import os

In [2]:
# Key and nonce (fixed nonce length)
key = os.urandom(32)             # 256-bit key
nonce = os.urandom(16)           # 128-bit nonce (required by this implementation)
plaintext = b"Tim Newsome said:In God we trust. Everybody else we verify using PGP!"  
# The b prefix denotes a byte string (or bytes object) 
# rather than a standard Unicode string (str).

Next, we initialize the ChaCha20 algorithm using our key and nonce, create a cipher context, and encrypt the plaintext.

In [3]:
# Encrypt
algorithm = algorithms.ChaCha20(key, nonce)
cipher = Cipher(algorithm, mode=None, backend=default_backend())
encryptor = cipher.encryptor()
ciphertext = encryptor.update(plaintext)

Note that once I have decided to use ChaCha20 algorithm, with a key and a nonce (Initialization vector), when we call this encription algorithm we must specify a mode and a backed:

1. mode (Encryption Mode)
What it means: Block ciphers (like AES) require a "mode of operation" (such as CBC, GCM, or ECB) to dictate how blocks of data are chained together and whether authentication is provided.

Why it is None here: ChaCha20 is a stream cipher, not a block cipher. It generates a continuous pseudo-random keystream that is combined directly with the plaintext via XOR. Because it does not operate on fixed-size blocks, traditional block modes do not apply, so mode=None is explicitly required.

2. backend (Cryptographic Backend)
What it means: The backend determines what software or hardware library executes the underlying cryptographic math.

Why default_backend() is used: Python's cryptography library separates its high-level API from the low-level C implementations. default_backend() tells the library to automatically select the fastest and most secure available implementation on your system (typically utilizing optimized C code via OpenSSL). It ensures compatibility without requiring the developer to manually configure system-specific drivers or libraries.

Because stream ciphers are symmetric, we can reverse the process using a fresh decryptor object to retrieve the original message.

In [4]:
# Decrypt
decryptor = cipher.decryptor()
recovered = decryptor.update(ciphertext)

In [5]:

print("Ciphertext:", ciphertext.hex())
print("Recovered:", recovered)


Ciphertext: 260b2faf2d6ed1223b159a7c15e599dd3a3ea2ad97bd71758a54188d82bdb75e4e23267da5450226322f78751bb69126b0c07e371ad91be79c9637493c5bac6cf838fa0def
Recovered: b'Tim Newsome said:In God we trust. Everybody else we verify using PGP!'


Finally, we define a helper function to view the data as binary strings and isolate the raw keystream using the XOR property: 

$$Keystream = Ciphertext \oplus Plaintext.$$
  

In [6]:
# Helper: convert bytes to continuous bit string
def to_bits(data: bytes) -> str:
    return ''.join(f'{byte:08b}' for byte in data)

# Recover the keystream: ciphertext ⊕ plaintext
keystream = bytes(c ^ p for c, p in zip(ciphertext, plaintext))

print("Plaintext  (bits):", to_bits(plaintext))
print("Ciphertext (bits):", to_bits(ciphertext))
print("Keystream  (bits):", to_bits(keystream))
print("Recovered  (bits):", to_bits(recovered))

Plaintext  (bits): 010101000110100101101101001000000100111001100101011101110111001101101111011011010110010100100000011100110110000101101001011001000011101001001001011011100010000001000111011011110110010000100000011101110110010100100000011101000111001001110101011100110111010000101110001000000100010101110110011001010111001001111001011000100110111101100100011110010010000001100101011011000111001101100101001000000111011101100101001000000111011001100101011100100110100101100110011110010010000001110101011100110110100101101110011001110010000001010000010001110101000000100001
Ciphertext (bits): 0010011000001011001011111010111100101101011011101101000100100010001110110001010110011010011111000001010111100101100110011101110100111010001111101010001010101101100101111011110101110001011101011000101001010100000110001000110110000010101111011011011101011110010011100010001100100110011111011010010101000101000000100010011000110010001011110111100001110101000110111011011010010001001001101011000011000000011111100

## We can collect all the above in a unique cell:

In [7]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms
from cryptography.hazmat.backends import default_backend
import os

# Key and nonce (fixed nonce length)
key = os.urandom(32)             # 256-bit key
nonce = os.urandom(16)           # 128-bit nonce (required by this implementation)
plaintext = b"Tim Newsome said:In God we trust. Everybody else we verify using PGP!"

# Encrypt
algorithm = algorithms.ChaCha20(key, nonce)
cipher = Cipher(algorithm, mode=None, backend=default_backend())
encryptor = cipher.encryptor()
ciphertext = encryptor.update(plaintext)

# Decrypt
decryptor = cipher.decryptor()
recovered = decryptor.update(ciphertext)

print("Ciphertext:", ciphertext.hex())
print("Recovered:", recovered)


# Helper: convert bytes to continuous bit string
def to_bits(data: bytes) -> str:
    return ''.join(f'{byte:08b}' for byte in data)

# Recover the keystream: ciphertext ⊕ plaintext
keystream = bytes(c ^ p for c, p in zip(ciphertext, plaintext))

print("Plaintext  (bits):", to_bits(plaintext))
print("Ciphertext (bits):", to_bits(ciphertext))
print("Keystream  (bits):", to_bits(keystream))
print("Recovered  (bits):", to_bits(recovered))

Ciphertext: b97ff8d5d9e6775b2578393fab873cff84765638718ad2c1b1526700ac73d61c2f822c9d7e6776cadf1b213c4022634e37b75b36e389df3cb18b8c29b23088faf93815dfd5
Recovered: b'Tim Newsome said:In God we trust. Everybody else we verify using PGP!'
Plaintext  (bits): 010101000110100101101101001000000100111001100101011101110111001101101111011011010110010100100000011100110110000101101001011001000011101001001001011011100010000001000111011011110110010000100000011101110110010100100000011101000111001001110101011100110111010000101110001000000100010101110110011001010111001001111001011000100110111101100100011110010010000001100101011011000111001101100101001000000111011101100101001000000111011001100101011100100110100101100110011110010010000001110101011100110110100101101110011001110010000001010000010001110101000000100001
Ciphertext (bits): 101110010111111111111000110101011101100111100110011101110101101100100101011110000011100100111111101010111000011100111100111111111000010001110110010101100011100001110001100010

## Salsa20 – Using `pycryptodome`

In [8]:
!pip install pycryptodome

DEPRECATION: pyodbc 4.0.0-unsupported has a non-standard version number. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pyodbc or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [9]:
from Crypto.Cipher import Salsa20
import os

# Key and nonce
key = os.urandom(32)              # 256-bit key
nonce = os.urandom(8)             # 64-bit nonce
plaintext = b"Tim Newsome said:In God we trust. Everybody else we verify using PGP!"

# Encrypt
cipher = Salsa20.new(key=key, nonce=nonce)
ciphertext = cipher.encrypt(plaintext)

# Decrypt
cipher_dec = Salsa20.new(key=key, nonce=nonce)
recovered = cipher_dec.decrypt(ciphertext)

print("Ciphertext:", ciphertext.hex())
print("Recovered:", recovered)

# Recover the keystream: ciphertext ⊕ plaintext
keystream = bytes(c ^ p for c, p in zip(ciphertext, plaintext))

print("Plaintext  (bits):", to_bits(plaintext))
print("Ciphertext (bits):", to_bits(ciphertext))
print("Keystream  (bits):", to_bits(keystream))
print("Recovered  (bits):", to_bits(recovered))

Ciphertext: 9c4a5db044da86245f0345c0b17f92904a824b1568bcddcbfe2fe7483a02116807a46fcd74d58241e433e946d8bc70e1f77ee6cec799163828987c578d53064e3a5a57e2d9
Recovered: b'Tim Newsome said:In God we trust. Everybody else we verify using PGP!'
Plaintext  (bits): 010101000110100101101101001000000100111001100101011101110111001101101111011011010110010100100000011100110110000101101001011001000011101001001001011011100010000001000111011011110110010000100000011101110110010100100000011101000111001001110101011100110111010000101110001000000100010101110110011001010111001001111001011000100110111101100100011110010010000001100101011011000111001101100101001000000111011101100101001000000111011001100101011100100110100101100110011110010010000001110101011100110110100101101110011001110010000001010000010001110101000000100001
Ciphertext (bits): 100111000100101001011101101100000100010011011010100001100010010001011111000000110100010111000000101100010111111110010010100100000100101010000010010010110001010101101000101111

## Trivium – Toy Python implementation

The Trivium stream cipher operates on a 288-bit internal state represented as a bit vector 
$$S = (s_0, s_1, \dots, s_{287}) \in \mathbb{Z}_2^{288}$$
The cipher's architecture is built upon three interconnected Non-Linear Feedback Shift Registers (NFSRs) of varying lengths, governed by strict algebraic update rules.

<b font='bold'> State Initialization</b> (trivium\_init)

The algorithm maps an 80-bit key $K = (k_0, \dots, k_{79})$ and an 80-bit Initialization Vector $IV = (iv_0, \dots, iv_{79})$ into the 288-bit state vector $S$ through the following discrete assignment:

$$\begin{aligned}
(s_0, s_1, \dots, s_{79}) &\leftarrow (k_0, k_1, \dots, k_{79}) \\
(s_{80}, s_{81}, \dots, s_{92}) &\leftarrow (0, 0, \dots, 0) \\
(s_{93}, s_{94}, \dots, s_{172}) &\leftarrow (iv_0, iv_1, \dots, iv_{79}) \\
(s_{173}, s_{174}, \dots, s_{284}) &\leftarrow (0, 0, \dots, 0) \\
(s_{285}, s_{286}, s_{287}) &\leftarrow (1, 1, 1)
\end{aligned}$$

<b font='bold'> The Warm-Up Phase</b> ($4 \times 288$ Clock Cycles)

Before generating any keystream, the cipher undergoes $1,152$ clock cycles ($4 \times 288$ cycles) of non-linear mixing without output generation to diffuse the key and IV bits across the entire register. In each clock cycle $t$, intermediate feedback bits $t_1, t_2, t_3 \in \mathbb{Z}_2$ are computed using quadratic Boolean functions:

$$\begin{aligned}
t_1 &= s_{66} \oplus (s_{91} \cdot s_{92}) \oplus s_{93} \oplus s_{171} \\
t_2 &= s_{162} \oplus (s_{175} \cdot s_{176}) \oplus s_{177} \oplus s_{264} \\
t_3 &= s_{243} \oplus (s_{286} \cdot s_{287}) \oplus s_{0} \oplus s_{69}
\end{aligned}$$

<b color='orange'>(Note: Indices correspond to 0-based Python indexing where Python's state[65] maps to the formal mathematical notation $s_{66}$ depending on indexing conventions, reflecting the shift register taps).</b>

The state vector undergoes a simultaneous synchronous shift:
$$S^{(t+1)} = \left( t_3, \, s_0, \dots, s_{91}, \, t_1, \, s_{93}, \dots, s_{175}, \, t_2, \, s_{177}, \dots, s_{286} \right)$$

<b font='bold'> Keystream Generation</b> (trivium\_keystream)

Once initialized, Trivium generates a pseudo-random keystream bit $z \in \mathbb{Z}_2$ per clock cycle. The output bit is a linear combination of taps spanning the three register sections:

$$z = (s_{65} \oplus s_{92}) \oplus (s_{161} \oplus s_{176}) \oplus (s_{242} \oplus s_{287})$$

Expressed formally, using intermediate tap variables:
$$\begin{aligned}
t_1 &= s_{65} \oplus s_{92} \\
t_2 &= s_{161} \oplus s_{176} \\
t_3 &= s_{242} \oplus s_{287} \\
z &= t_1 \oplus t_2 \oplus t_3
\end{aligned}$$

Immediately after extracting $z$, the register updates its feedback paths with non-linear terms to prepare for the next cycle:
$$\begin{aligned}
t_1 &\leftarrow t_1 \oplus (s_{90} \cdot s_{91}) \oplus s_{170} \\
t_2 &\leftarrow t_2 \oplus (s_{174} \cdot s_{175}) \oplus s_{263} \\
t_3 &\leftarrow t_3 \oplus (s_{285} \cdot s_{286}) \oplus s_{68}
\end{aligned}$$

The entire 288-bit register shifts forward:
$$S \leftarrow [t_3] + S[0:92] + [t_1] + S[93:176] + [t_2] + S[177:287]$$

This process repeats for $8 \times L$ cycles (where $L$ is the length of the plaintext in bytes).

<b font='bold'> Symmetric Encryption and Decryption</b> (xor\_bytes)

Trivium functions as a synchronous additive stream cipher. The keystream bit sequence $Z = (z_0, z_1, z_2, \dots)$ is grouped into bytes $z^{(byte)}_n = \sum_{i=0}^{7} z_{8n + i} 2^i$.

Encryption and decryption are identical algebraic operations defined by the bitwise XOR ($\oplus$) group action over $\mathbb{Z}_2^8$:

$$\text{Ciphertext Byte: } c_n = p_n \oplus z^{(byte)}_n$$

$$\text{Recovered Plaintext Byte: } p_n = c_n \oplus z^{(byte)}_n$$

Because XOR is its own inverse ($x \oplus y \oplus y = x$), passing the ciphertext through the keystream generated by an identically re-initialized state (trivium\_init(key, iv)) perfectly annihilates the keystream, algebraically yielding the original plaintext.

The following script shows an implementation of Trivium, and its use on a short plaintext. 

In [11]:
def trivium_init(key, iv):
    # Trivium uses 80-bit key and IV
    assert len(key) == 10 and len(iv) == 10
    key_bits = sum([[int(b) >> i & 1 for i in range(8)] for b in key], [])
    iv_bits  = sum([[int(b) >> i & 1 for i in range(8)] for b in iv], [])

    state = [0] * 288
    state[:80] = key_bits
    state[93:173] = iv_bits
    state[285:] = [1, 1, 1]

    for _ in range(4 * 288):  # Warm-up phase
        t1 = state[65] ^ (state[90] & state[91]) ^ state[92] ^ state[170]
        t2 = state[161] ^ (state[174] & state[175]) ^ state[176] ^ state[263]
        t3 = state[242] ^ (state[285] & state[286]) ^ state[287] ^ state[68]
        state = [t3] + state[:92] + [t1] + state[93:176] + [t2] + state[177:287]
    return state

def trivium_keystream(state, length):
    stream = []
    for _ in range(length * 8):
        t1 = state[65] ^ state[92]
        t2 = state[161] ^ state[176]
        t3 = state[242] ^ state[287]
        z = t1 ^ t2 ^ t3
        stream.append(z)
        t1 = t1 ^ (state[90] & state[91]) ^ state[170]
        t2 = t2 ^ (state[174] & state[175]) ^ state[263]
        t3 = t3 ^ (state[285] & state[286]) ^ state[68]
        state = [t3] + state[:92] + [t1] + state[93:176] + [t2] + state[177:287]
    return stream

def xor_bytes(data, stream_bits):
    return bytes(b ^ sum([bit << i for i, bit in enumerate(stream_bits[n*8:n*8+8])])
                 for n, b in enumerate(data))

# Key, IV (10 bytes each = 80 bits)
key = b'1234567890'
iv  = b'abcdefghij'
plaintext = b"Tim Newsome said:In God we trust. Everybody else we verify using PGP!"

# Initialize and encrypt
state = trivium_init(key, iv)
stream_bits = trivium_keystream(state.copy(), len(plaintext))
ciphertext = xor_bytes(plaintext, stream_bits)

# Decrypt
state = trivium_init(key, iv)
stream_bits = trivium_keystream(state.copy(), len(ciphertext))
recovered = xor_bytes(ciphertext, stream_bits)

print("Ciphertext:", ciphertext.hex())
print("Recovered:", recovered)

# Recover the keystream: ciphertext ⊕ plaintext
keystream = bytes(c ^ p for c, p in zip(ciphertext, plaintext))

print("Plaintext  (bits):", to_bits(plaintext))
print("Ciphertext (bits):", to_bits(ciphertext))
print("Keystream  (bits):", to_bits(keystream))
print("Recovered  (bits):", to_bits(recovered))

Ciphertext: 03f75bd3194fc487b7f4afa9891c9cc544df46afd1901722533b37ce59bed656e4e261022eb6b4d9d7d1cffac8d2a56f68076d2c89dcd5ac1fbd0753bedc5fa13d33ba8a53
Recovered: b'Tim Newsome said:In God we trust. Everybody else we verify using PGP!'
Plaintext  (bits): 010101000110100101101101001000000100111001100101011101110111001101101111011011010110010100100000011100110110000101101001011001000011101001001001011011100010000001000111011011110110010000100000011101110110010100100000011101000111001001110101011100110111010000101110001000000100010101110110011001010111001001111001011000100110111101100100011110010010000001100101011011000111001101100101001000000111011101100101001000000111011001100101011100100110100101100110011110010010000001110101011100110110100101101110011001110010000001010000010001110101000000100001
Ciphertext (bits): 000000111111011101011011110100110001100101001111110001001000011110110111111101001010111110101001100010010001110010011100110001010100010011011111010001101010111111010001100100